In [1]:
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# --- Préparation / modélisation ---
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_val_predict,
    StratifiedKFold, KFold
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# --- Objectif B : classification (poste) ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# --- Objectif C : régression (buts vs xG) ---
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# --- Objectif A : clustering (profils de joueurs) ---
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# --- Objectif D : similarité entre joueurs ---
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment

pd.set_option("display.max_columns", 300)   # sinon pandas cache la plupart de vos 267 colonnes
pd.set_option("display.max_rows", 100)

In [2]:
df = pd.read_csv("players_data-2024_2025.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'players_data-2024_2025.csv'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
#missing values as a percentage
(df.isna().mean() * 100).sort_values(ascending=False).head(20).round(1)

In [ ]:
df.duplicated().sum()

In [ ]:
df_clean = df.copy()
df_clean.shape

In [ ]:
print("Lignes totalement dupliquées :", df_clean.duplicated().sum())
print("Doublons Player + Squad      :", df_clean.duplicated(subset=["Player", "Squad"]).sum())
print("Joueurs sur plusieurs lignes :", (df_clean["Player"].value_counts() > 1).sum())

In [ ]:
#supprimer les colonnes redondantes
redondantes = []
for c in df_clean.columns:
    if "_stats_" in c:
        base = c.split("_stats_")[0]
        if base in df_clean.columns:
            m = df_clean[c].notna()
            if df_clean.loc[m, c].equals(df_clean.loc[m, base]):
                redondantes.append(c)

rk = [c for c in df_clean.columns if c.startswith("Rk")]   # colonnes de rang, inutiles
df_clean = df_clean.drop(columns=list(set(redondantes + rk)))
df_clean.shape

In [ ]:
#nett format
# "eng ENG" -> "ENG"   |   "eng Premier League" -> "Premier League"
df_clean["Nation_code"] = df_clean["Nation"].str.split(" ").str[-1]
df_clean["League"] = df_clean["Comp"].str.split(" ", n=1).str[1]

df_clean[["Nation", "Nation_code", "Comp", "League"]].head()

In [ ]:
#vm diagnostic
na = pd.DataFrame({
    "n_manquants": df_clean.isna().sum(),
    "pct": (df_clean.isna().mean() * 100).round(1)
})
na[na["n_manquants"] > 0].sort_values("pct", ascending=False).head(30)

In [ ]:
gardiens = df_clean[df_clean["Pos"] == "GK"].copy()
joueurs  = df_clean[df_clean["Pos"] != "GK"].copy()

# joueurs de champ : on retire les colonnes 100 % vides (stats de gardien)
joueurs = joueurs.dropna(axis=1, how="all")

# gardiens : on retire les colonnes quasi vides (stats de tir des joueurs de champ)
gardiens = gardiens.loc[:, gardiens.isna().mean() < 0.9]

print("Joueurs de champ :", joueurs.shape)
print("Gardiens         :", gardiens.shape)

In [ ]:
# Quelques lignes sans nationalité / âge / année de naissance
df_clean["Nation_code"] = df_clean["Nation_code"].fillna("Unknown")
df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median()).astype(int)

In [ ]:
ratios = ["SoT%", "G/Sh", "G/SoT", "npxG/Sh", "Dist", "Succ%", "Tkld%",
          "Tkl%", "Won%", "Cmp%", "Mn/Start", "Mn/Sub"]
joueurs[ratios] = joueurs[ratios].fillna(0)

In [ ]:
na = joueurs.isna().sum()
na[na > 0]

In [ ]:
print("Minutes > 3420 (38 matchs x 90) :", (df_clean["Min"] > 38 * 90).sum())
print("Age < 15 ou > 45                 :", ((df_clean["Age"] < 15) | (df_clean["Age"] > 45)).sum())

pct = [c for c in df_clean.columns if c.endswith("%")]
print("Pourcentages > 100               :", (df_clean[pct] > 100).sum().sum())

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    i = q3 - q1
    return (s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)

cols = ["Min", "Gls", "Ast", "xG", "Sh", "Tkl", "Touches"]
pd.DataFrame({c: [iqr_outliers(joueurs[c]).sum()] for c in cols}, index=["n_outliers"]).T

In [ ]:
joueurs.loc[iqr_outliers(joueurs["Gls"]), ["Player", "Squad", "Gls", "xG"]] \
       .sort_values("Gls", ascending=False).head(10)

In [ ]:
import matplotlib.pyplot as plt

joueurs[cols].plot(kind="box", subplots=True, layout=(2, 4),
                   figsize=(14, 7), sharex=False, sharey=False)
plt.tight_layout()
plt.show()

In [ ]:
joueurs_900 = joueurs[joueurs["Min"] >= 900].copy()   # ≈ 10 matchs complets
print(joueurs.shape, "->", joueurs_900.shape)

In [ ]:
cumul = ["MP", "Starts", "Min", "90s", "Gls", "Ast", "G+A", "G-PK", "PK", "PKatt",
         "CrdY", "CrdR", "xG", "npxG", "xAG", "PrgC", "PrgP", "PrgR"]

df_unique = (df_clean
    .groupby(["Player", "Born"], dropna=False)
    .agg({**{c: "sum" for c in cumul},
          "Squad": lambda s: " / ".join(s),
          "Pos": "first", "League": "first", "Age": "first", "Nation_code": "first"})
    .reset_index())

df_unique.shape        # (2707, 25)

In [ ]:
#predire le poste
data = joueurs[joueurs["Min"] >= 900].copy()
data["Pos_main"] = data["Pos"].str.split(",").str[0]
data["Pos_main"].value_counts()

In [ ]:
cnt = ["Gls","Ast","xG","xAG","Sh","SoT","KP","PPA","PrgP","PrgC","PrgR","Tkl","TklW","Int",
       "Blocks","Clr","Touches","Carries","Succ","Cmp","Att","Crs","SCA","GCA","Recov","Won",
       "Lost","Fls","Fld","TB","Sw","Att Pen","Def Pen","Att 3rd_stats_possession",
       "Def 3rd_stats_possession","Mis","Dis","Err","Off","CrdY","TI","CK","1/3","CPA",
       "CrsPA","Rec"]

X = data[cnt].div(data["90s"], axis=0)
X.columns = [c + "_90" for c in cnt]
y = data["Pos_main"]
X.shape

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
modeles = {
    "Régression logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Arbre de décision": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
}

for nom, m in modeles.items():
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    print(f"=== {nom} : accuracy test = {accuracy_score(yte, p):.3f}")
    print(classification_report(yte, p))

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
for nom, m in modeles.items():
    print(nom, cross_val_score(m, X, y, cv=cv).mean().round(3))

In [ ]:
#arbre de décision : choix de la profondeur maximale
for d_ in [2, 3, 4, 5, 6, 8, 10, None]:
    dt_test = DecisionTreeClassifier(max_depth=d_, class_weight="balanced", random_state=42)
    acc = cross_val_score(dt_test, X, y, cv=cv, scoring="accuracy").mean()
    print(f"max_depth = {str(d_):5s} -> accuracy CV = {acc:.3f}")

In [ ]:
#arbre de décision : matrice de confusion et structure de l'arbre
dt = modeles["Arbre de décision"]

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(dt, Xte, yte, labels=["DF", "MF", "FW"], cmap="Blues", ax=ax)
plt.title("Arbre de décision")
plt.show()

plt.figure(figsize=(20, 8))
plot_tree(dt, feature_names=X.columns, class_names=dt.classes_, filled=True, max_depth=3, fontsize=8)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(modeles["Régression logistique"], Xte, yte,
                                      labels=["DF", "MF", "FW"], cmap="Blues", ax=ax)
plt.show()

In [ ]:
rf = modeles["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.head(15).sort_values().plot(kind="barh", figsize=(7, 5), title="Variables les plus discriminantes")
plt.show()

In [ ]:
pred = cross_val_predict(modeles["Random Forest"], X, y, cv=cv)
erreurs = data.assign(Pos_predite=pred).loc[pred != y, ["Player", "Squad", "Pos", "Pos_main", "Pos_predite", "Min"]]
print(len(erreurs), "joueurs sur", len(data))
erreurs.head(15)

In [ ]:
#sur- et sous-performance des buteurs
d = joueurs[joueurs["Sh"] >= 20].copy()
d["npG"] = d["G-PK"]
d.shape

In [ ]:
d["diff"] = d["npG"] - d["npxG"]              # buts marqués - buts attendus
d["z"] = d["diff"] / np.sqrt(d["npxG"])       # écart en "unités de hasard"
d[["npG", "npxG", "diff", "z"]].describe().T.round(2)

In [ ]:
lr = LinearRegression().fit(d[["npxG"]], d["npG"])
d["npG_pred"] = lr.predict(d[["npxG"]])
print("coefficient :", lr.coef_[0].round(2), "| constante :", lr.intercept_.round(2))

cv = KFold(5, shuffle=True, random_state=42)
print("R² (validation croisée) :", cross_val_score(LinearRegression(), d[["npxG"]], d["npG"], cv=cv, scoring="r2").mean().round(3))

In [ ]:
essais = {
    "npxG seul": ["npxG"],
    "npxG + SoT": ["npxG", "SoT"],
    "npxG + SoT + Dist + Sh": ["npxG", "SoT", "Dist", "Sh"],
}
for nom, f in essais.items():
    r2  = cross_val_score(LinearRegression(), d[f], d["npG"], cv=cv, scoring="r2").mean()
    mae = -cross_val_score(LinearRegression(), d[f], d["npG"], cv=cv, scoring="neg_mean_absolute_error").mean()
    print(f"{nom:26s} R² = {r2:.3f}   erreur moyenne = {mae:.2f} buts")

In [ ]:
#la plus grand sous performant
cols = ["Player", "Squad", "League", "Pos", "Sh", "npxG", "npG", "diff", "z"]

print("SUR-PERFORMEURS (plus de buts que prévu)")
display(d.sort_values("z", ascending=False)[cols].head(10).round(2))

print("SOUS-PERFORMEURS (moins de buts que prévu)")
display(d.sort_values("z")[cols].head(10).round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=d, x="npxG", y="npG", hue="League", alpha=0.7, ax=ax)

lim = d[["npxG", "npG"]].max().max()
ax.plot([0, lim], [0, lim], "k--", lw=1, label="npG = npxG")

for _, r in pd.concat([d.nlargest(5, "z"), d.nsmallest(3, "z")]).iterrows():
    ax.annotate(r["Player"], (r["npxG"], r["npG"]), fontsize=8)

ax.set_title("Buts hors penalty vs buts attendus")
ax.legend()
plt.show()

In [ ]:
print("Écart-type de z :", round(d["z"].std(), 2))
print("Joueurs avec z > 2  :", (d["z"] > 2).sum())
print("Joueurs avec z < -2 :", (d["z"] < -2).sum())

print("\nÉcart moyen (buts - xG) par ligue :")
print(d.groupby("League")["diff"].mean().round(2))

print("\nCorrélation de l'écart avec :")
print(d[["diff", "SoT%", "Dist"]].corr()["diff"].round(2))

In [ ]:
#objectif a : Variables par 90 minutes
data = joueurs_900.copy().reset_index(drop=True)
data["Pos_main"] = data["Pos"].str.split(",").str[0]   # sert seulement à comparer, pas à clusteriser

cnt = ["Gls","Ast","xG","xAG","Sh","SoT","KP","PPA","PrgP","PrgC","PrgR","Tkl","TklW","Int",
       "Blocks","Clr","Touches","Carries","Succ","Cmp","Att","Crs","SCA","GCA","Recov","Won",
       "Lost","Fls","Fld","TB","Sw","Att Pen","Def Pen","Att 3rd_stats_possession",
       "Def 3rd_stats_possession","Mis","Dis","Err","Off","CrdY","TI","CK","1/3","CPA",
       "CrsPA","Rec"]

X = data[cnt].div(data["90s"], axis=0)
X.columns = [c + "_90" for c in cnt]
X.shape          # (1450, 46)

In [ ]:
scaler = StandardScaler()
Xs = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

In [ ]:
pca_full = PCA().fit(Xs)
cumul = np.cumsum(pca_full.explained_variance_ratio_)

plt.plot(range(1, len(cumul) + 1), cumul, "o-")
plt.axhline(0.8, color="red", ls="--")
plt.xlabel("Nombre de composantes"); plt.ylabel("Variance expliquée cumulée")
plt.show()

print("Composantes pour 80 % de variance :", np.argmax(cumul >= 0.8) + 1)

In [ ]:
pca = PCA(n_components=10, random_state=42)
Z = pca.fit_transform(Xs)

In [ ]:
ks = range(2, 11)
inertie, silh = [], []
for k in ks:
    m = KMeans(n_clusters=k, n_init=20, random_state=42).fit(Z)
    inertie.append(m.inertia_)
    silh.append(silhouette_score(Z, m.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ks, inertie, "o-"); ax[0].set_title("Méthode du coude"); ax[0].set_xlabel("k")
ax[1].plot(ks, silh, "o-");    ax[1].set_title("Silhouette");       ax[1].set_xlabel("k")
plt.show()

In [ ]:
km = KMeans(n_clusters=6, n_init=20, random_state=42).fit(Z)
data["cluster"] = km.labels_
print("Silhouette :", round(silhouette_score(Z, km.labels_), 3))
data["cluster"].value_counts().sort_index()

In [ ]:
profil = Xs.groupby(data["cluster"]).mean()     # profil moyen de chaque cluster

signature = {"Défenseur central": "Clr_90", "Attaquant finisseur": "Sh_90",
             "Latéral / piston": "TI_90", "Créateur / ailier": "CPA_90",
             "Milieu récupérateur": "Tkl_90", "Milieu relayeur": "PrgP_90"}
noms = list(signature)

M = np.array([[profil.loc[c, signature[n]] for n in noms] for c in profil.index])
lignes, colonnes = linear_sum_assignment(-M)     # meilleure association 1 cluster ↔ 1 nom
map_noms = {profil.index[i]: noms[j] for i, j in zip(lignes, colonnes)}

data["Profil"] = data["cluster"].map(map_noms)
data["Profil"].value_counts()

In [ ]:
cles = ["Clr_90","Def Pen_90","Tkl_90","Recov_90","TI_90","Crs_90","PrgP_90","1/3_90",
        "Touches_90","CPA_90","PrgC_90","Succ_90","SCA_90","xAG_90","Sh_90","xG_90","Gls_90","Att Pen_90","Won_90"]
prof_cles = Xs.groupby(data["Profil"])[cles].mean()

plt.figure(figsize=(13, 4.5))
sns.heatmap(prof_cles, cmap="RdBu_r", center=0, annot=True, fmt=".1f")
plt.title("Profil moyen (écarts-types par rapport à la moyenne)")
plt.show()

In [ ]:
pd.crosstab(data["Profil"], data["Pos_main"])

In [ ]:
Z2 = PCA(n_components=2, random_state=42).fit_transform(Xs)

fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(x=Z2[:, 0], y=Z2[:, 1], hue=data["Profil"], alpha=0.7, ax=ax)
ax.set_xlabel("Composante 1 (35 % de variance)")
ax.set_ylabel("Composante 2 (19 % de variance)")
plt.show()

In [ ]:
#objectif d : joueurs similaires
def normaliser(s):
    """minuscules et sans accents, pour que 'mbappe' trouve 'Mbappé'"""
    return "".join(ch for ch in unicodedata.normalize("NFD", str(s).lower())
                   if unicodedata.category(ch) != "Mn")

data["_nom"] = data["Player"].map(normaliser)
sim = cosine_similarity(Xs.values)      # matrice 1450 x 1450
sim.shape

In [ ]:
def joueurs_similaires(nom, n=5, meme_ligue=False, poste=None):
    q = normaliser(nom).strip()

    exact = data.index[data["_nom"] == q]
    if len(exact) > 0:
        i = data.loc[exact, "Min"].idxmax()
    else:
        cand = data.index[data["_nom"].str.contains(q, regex=False)]
        if len(cand) == 0:
            print(f"Aucun joueur '{nom}' (moins de 900 minutes ?)")
            return None
        if len(cand) > 1:
            print("Plusieurs joueurs correspondent, précisez :")
            print(data.loc[cand, ["Player", "Squad"]].to_string(index=False))
            return None
        i = cand[0]

    s = pd.Series(sim[i], index=data.index)
    s = s[data["Player"] != data.loc[i, "Player"]]          # exclut le joueur lui-même
    if meme_ligue:
        s = s[data.loc[s.index, "League"] == data.loc[i, "League"]]
    if poste:
        s = s[data.loc[s.index, "Pos_main"] == poste]

    top = s.sort_values(ascending=False).head(n)
    print(f"Joueur : {data.loc[i, 'Player']} ({data.loc[i, 'Squad']}) - profil : {data.loc[i, 'Profil']}")
    res = data.loc[top.index, ["Player", "Squad", "League", "Pos", "Age", "Min", "Profil"]].copy()
    res.insert(0, "Similarité", top.round(3))
    return res.reset_index(drop=True)

In [ ]:
joueurs_similaires("mbappe")

In [ ]:
def radar(a, b):
    feats = ["Sh_90","xG_90","SCA_90","PrgC_90","PrgP_90","Tkl_90","Int_90","Clr_90","Won_90"]
    ang = np.linspace(0, 2 * np.pi, len(feats), endpoint=False).tolist()
    ang += ang[:1]
    fig, ax = plt.subplots(subplot_kw={"polar": True}, figsize=(6.5, 6.5))
    for nom in (a, b):
        i = data.index[data["Player"] == nom][0]
        v = Xs.loc[i, feats].tolist(); v += v[:1]
        ax.plot(ang, v, label=nom); ax.fill(ang, v, alpha=0.15)
    ax.set_xticks(ang[:-1]); ax.set_xticklabels(feats)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.show()

radar("Lamine Yamal", "Bradley Barcola")

In [ ]:
mm_profil, mm_poste = [], []
for i in range(len(data)):
    s = sim[i].copy(); s[i] = -9
    voisins = np.argsort(-s)[:5]
    mm_profil.append((data.loc[voisins, "Profil"].values == data.loc[i, "Profil"]).mean())
    mm_poste.append((data.loc[voisins, "Pos_main"].values == data.loc[i, "Pos_main"]).mean())

print("Voisins de même profil :", round(np.mean(mm_profil), 3))
print("Voisins de même poste  :", round(np.mean(mm_poste), 3))